In [27]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd

In [28]:
prison = Prison()
actions = prison.Get_Actions()

strategies = {
    0: Random_Strategy(0, actions),
    1: Always_Cooperate(1, actions),
    2: Always_Betray(2, actions),
    3: Random_Strategy(3, actions, p_coop=0.1),
    4 : Patient_Unforgiving(4, actions),
}

In [29]:
gm = Game_Master(prison, strategies=strategies, duel_size=2)
duel_matrix, rewards = gm.Tournament(4, Game_Master.Game_Type.All_Vs_All, True)
rewards.Sort_Total_Rewards()

In [30]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

## Wyniki

In [31]:
name_total_reward_df = pd.DataFrame.from_dict(total_rewards_per_name, orient="index", columns=["Total Reward"])
name_total_reward_df.index.name = "Strategy Name"
name_total_reward_df

,Total Reward
Strategy Name,
Always_Betray,44
Random_Strategy (p_coop=0.1),42
Patient_Unforgiving (patience=1),33
Random_Strategy (p_coop=0.5),27
Always_Cooperate,18


In [32]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

In [33]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [ ]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

,Random_Strategy (p_coop=0.5),Always_Cooperate,Always_Betray,Random_Strategy (p_coop=0.1),Patient_Unforgiving (patience=1)
Random_Strategy (p_coop=0.5),"(0, 0)","(16, 6)","(2, 12)","(2, 12)","(7, 7)"
Always_Cooperate,"(6, 16)","(0, 0)","(0, 20)","(0, 20)","(12, 12)"
Always_Betray,"(12, 2)","(20, 0)","(0, 0)","(4, 4)","(8, 3)"
Random_Strategy (p_coop=0.1),"(12, 2)","(20, 0)","(4, 4)","(0, 0)","(6, 11)"
Patient_Unforgiving (patience=1),"(7, 7)","(12, 12)","(3, 8)","(11, 6)","(0, 0)"


In [ ]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")